In [1]:
import glob
import itertools
import pandas as pd
import json
import re

In [2]:
json_paths = "/data/llm_icardio/All_studies_JSON_V2"
meta = {'studies':[]}#, 'dicoms':[]}
for path in glob.glob(f"{json_paths}/**.json"):
    with open(path) as f:
        print(path)
        data = json.load(f)
        meta['studies'] += data['studies']
        # meta['dicoms'] += data['dicoms']

/data/llm_icardio/All_studies_JSON_V2/study_info_9_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_13_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_10_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_3_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_1_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_11_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_7_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_15_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_14_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_16_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_2_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_8_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_6_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_12_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_5_of_16.json
/data/llm_icardio/All_studies_JSON_V2/study_info_4_of_16.json


In [3]:
df = pd.read_csv("/data/icardio/subsets/full_3DRL_subset_norm/subset_official_splits.csv", low_memory=False)

df = df[df['split_official_test'] == "test"]

In [4]:
studies_list = list(set(df['study'].to_list()))
filtered_studies = [
    d for d in meta['studies'] if d.get('study_uuid') in studies_list
]

In [11]:
pathos = {}
healthy = 0
regur = set()
for study in filtered_studies:
    h = 1
    if study['characterizations'] is None:
        continue
    for idx, charact in enumerate(study['characterizations']):
        found = [sub for sub in study['stratifications'] if sub in charact]
        found = found[0] if len(found) == 1 else "No Stratification"
        pattern = r"\s*{}\s*".format(re.escape(found))
        cleaned = re.sub(pattern, " ", charact)
        cleaned  = cleaned.strip()
        if found == "Normal" or found == "No Stratification":
            continue
        h = 0
        if cleaned == "Left Ventricular Hypertrophy" or cleaned == "Left Ventricular Enlargement" or cleaned == "Left Ventricular Basal Septal Hypertrophy":
            regur.add(study["study_uuid"]) #+'/'+found)
        # pathos[cleaned] = list(set(pathos.get(cleaned, []) + [found]))
        if cleaned not in pathos:
            pathos[cleaned] = {
                "strat": set(),
                "count": 0
            }
        # update set of found substrings
        pathos[cleaned]["strat"].add(found)
        # increment count
        pathos[cleaned]["count"] += 1

    healthy += h
print("Number of only healthy:", healthy)
pathos
regur

Number of only healthy: 10


{'st-0669-C8DE-B222',
 'st-0C93-8E7D-9CEB',
 'st-245D-613F-4DB2',
 'st-29DA-AD6F-07AA',
 'st-3EF9-2783-5CDE',
 'st-593C-4631-89E3',
 'st-6088-5334-D088',
 'st-6204-42D3-C00C',
 'st-66B5-029A-CE0E',
 'st-6B6F-A44B-5FA7',
 'st-7CEC-D71E-B63C',
 'st-7EB4-9A94-C9CC',
 'st-8620-1218-F2B2',
 'st-A234-2415-5724',
 'st-A9BF-2477-37D0',
 'st-B6C1-5159-ABB0',
 'st-B987-CFC5-9F29',
 'st-C139-F262-05BA',
 'st-C19D-EB20-4041',
 'st-CCB9-0946-EA0F',
 'st-D3F8-BFA9-D55A',
 'st-ED33-7269-CB60',
 'st-FEEB-CD15-98D0'}

In [6]:
for cond, info in pathos.items():
    strat = "/".join(sorted(info["strat"]))
    print(f"{cond}: {info['count']} [{strat}]")


Mitral Valvular Regurgitation: 92 [Mild/Moderate/Severe/Trace]
Tricuspid Regurgitation: 62 [Mild/Moderate/Severe]
Aortic Valvular Calcification: 6 [Mild/Moderate]
Aortic Root Dilation: 4 [Present]
Left Atrial Enlargement: 27 [Mild/Moderate/Severe]
Left Ventricular Hypertrophy: 15 [Borderline/Mild/Moderate/Severe]
Tricuspid Valvular Regurgitation: 39 [Trace]
Left Ventricular Basal Septal Hypertrophy: 6 [Mild/Severe]
Left Ventricular Enlargement: 2 [Moderate/Severe]
Aortic Valvular Insufficiency: 22 [Mild/Moderate/Severe/Trace]
Aortic Valvular Stenosis: 7 [Mild/Moderate/Severe]
Left Ventricular  Diastolic Dysfunction: 6 [Stage 1]
Aortic Root Dilated Ascending Aorta: 1 [Present]
Pericardial Fat Pad: 1 [Present]
Right Atrial Enlargement: 3 [Moderate/Severe]
Aortic Valvular Sclerosis: 4 [Mild/Moderate]
Pericardial Effusion: 1 [Present]
Pacemaker in Right Ventricle: 3 [Present]
Right Atrial Pacemaker: 2 [Present]
Left Ventricular Any Hypokinesis: 1 [Present]
